# 整体架构：  
$Encoder(负责提取特征)----->Decoder(利用特征进行预测)$

## Encoder

### Attention 注意力机制： 

每一个词或每一个特征点都会与上下文进行权重加和计算，得到一个新的融合了上下文信息的向量

序列中的每一个词或者图中的每一个特征点都会映射成向量 $X_n$  

经由可训练的权重矩阵 $W$ 将$X$,映射成各自的 $Q$, $K$, $V$  $(X, Q, K, V的维度都相同，因为要做残差连接)$ 

$Q$ = $X$ $\times$ $W_q$  
$K$ = $X$ $\times$ $W_k$  
$V$ = $X$ $\times$ $W_v$ 

每次反向传播更新的参数也只有 $W_q$, $W_k$, $W_v$  
而损失则是根据预测结果与实际标签的差距来确定

注意力分数计算：  
$\text Scale Dot-Product Attention$ 计算公式：  
$$
\text{AttentionScore} = \text{softmax}\left(\frac{Q \times K^T}{\sqrt{d_k}}\right)
$$

$d_k$:向量维度，引入向量维度消除维度对于训练稳定性的影响，防止在高纬度下，softmax梯度消失
  
Attention整体计算流程：  
1. 每个词的$Q$都会跟每一个词的$K$计算得分  
2. 得分 / $\sqrt{d_k}$
3. softmax后得到整个加权结果
4. $z_1 = s_1 \times v_1 + s_2 \times v_2 + ..... s_n \times v_n$ 
5. 统一时间计算出所有词的表示结果

### multi-headed 多头机制：

多头的本质是让模型在多个不同的表示子空间里并行地学习不同的注意力模式

$经典平均分割$：
- 同一个向量$X$，经由多个随机初始化的权重矩阵${W_n^q},{W_n^k},{W_n^v}$得到多组$q,k,v$形成多头，得到的注意力结果不同，得到的特征向量表达也不同

例子：  
输入 X 是 512 维向量  
每个头有自己的三个权重矩阵
8 个头并行做注意力，输出各自是 64 维，拼起来回到 512 维，再过一个输出投影。


多头注意力机制计算流程：
1. 通过不同的$head$得到多个特征表达  
2. 将所有特征拼接在一起
3. 在通过一层全连接层进行降维

### 位置信息表达：

Transformer对于位置信息并不敏感  
你打我 vs 我打你  
在Transformer中都有同一组qkv进行表示，但是在实际语义中二者天差地别，所以在Transformer中引入了位置信息的表达

### Transformer由多层堆叠
编码器的一层 = Self-Attention + FC(全连接层)

## Decoder

decoder从目标序列的embedding+位置编码来投影出每个向量qkv  
1. 通过遮罩自注意力只能看到前文的，输出一个融合前文的对V进行加权求和后的新向量
2. 通过交叉注意力来结合encoder提取的特征进行预测，拿着decoder的q去找encoder的输出kv,最后输出encoder各位置v向量的加权和
3. 通过前馈神经网络整理前面两步聚合来的信息

# 手撕Transformer

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
import math

#### 自注意力机制

In [7]:
class SelfAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)  # 代表对10%的神经元做一个随机失活，防止过拟合
        self.softmax = nn.Softmax(dim=-1)  # 将得分转换成概率分布，在最后一个维度进行归一化处理
    
    def forward(self, Q, K, V, mask=None):
        # X:batch, seq_len, d_model
        # batch: 一次送到模型的序列个数；seq_len: 一个序列中token的数量；d_model: embedding向量的维度
        # Q,query向量  维度: batch, heads, seq_len_q, d_k
        # K,key向量  维度: batch, heads, seq_len_k, d_k
        # V,value向量  维度: batch, heads, seq_len_v, d_v
        # mask 告诉模型哪些位置需要看，哪些不需要看
        d_k = Q.size(-1)  # q的最后一维是每个query向量的维度，代表对每个query进行缩放
        # batch, heads, seq_len_q, d_k * batch, heads, d_k, seq_len_k ->  batch, heads, seq_len_q, seq_len_k
        socres = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # 进行缩放 让模型训练的梯度更加稳定
        # 如果提供了mask，则通过mask==0来找到需要屏蔽的位置，masked_fill会将这些位置的值改为-inf(负无穷)
        # 进过softmax之后这些位置的值会变成0（被忽略）
        # 设置mask=0 表示被屏蔽  mask=1则当前位置可见
        if mask is not None:
            scores = scores.masked_fill(mask==0, float('-inf'))
        # batch, heads, seq_len_q, seq_len_k 对最后一维进行softmax，即对key进行，得到注意力权重矩阵，对每一个query的key权重之和为1
        attn = self.softmax(scores)
        attn = self.dropout(attn)  # 对注意力权重进行dropout,防止过拟合
        # attn:batch, heads, seq_len_q, seq_len_k；V:batch, heads, seq_len_v, d_v -> batch, heads, seq_len_q, d_v
        out = torch.matmul(attn, V)

        return out, attn

#### 多头注意力机制

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        # d_model embedding的维度 512
        # n_heads 为多头注意力的头数 8
        # d_model 需要能被 n_heads 整除 结果为64
        assert d_model % n_heads ==0
        self.d_k = d_model // n_heads  # 每个头的维度
        self.n_heads = n_heads

        # 将输入映射到q, k, v三个向量
        # 通过线性映射让模型具有学习能力
        self.W_q = nn.Linear(d_model, d_model)  # query的线性映射，维度不需要改变，方便后续的多头拆分
        self.W_k = nn.Linear(d_model, d_model)  # key的线性映射，维度不需要改变，方便后续的多头拆分
        self.W_v = nn.Linear(d_model, d_model)  # key的线性映射，维度不需要改变，方便后续的多头拆分
        self.fc = nn.Linear(d_model, d_model)  # 多头拼接后再映射回原来的d_model, 让模型融合不同头的信息

        self.attention = SelfAttention(dropout)  # 使用定义好的selfatten
        self.dropout = nn.Dropout(dropout)  # 防止过拟合
        self.norm = nn.LayerNorm(d_model)  # 用于残差后的归一化

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)  # 获取batch的大小
        # Q的维度 batch, seq_len, d_model -> batch, seq_len, self.n_heads, self.d_k -> batch, self.n_heads, seq_len, self.d_k
        # 为了让每个注意力头独立处理整个序列，方便后续计算注意力权重
        Q = self.W_q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        # 计算注意力
        out, attn = self.attention(Q, K, V, mask)  # attn为注意力权重，out为注意力加权后的值
        # batch, heads, seq_len_q, d_v -> batch, seq_len_q, heads, d_v
        # contiguous的目的是让tensor在内存中连续存储，避免view的时候产生报错
        # 多头的拼接
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads*self.d_k)  # out: batch, seq_len, d_model
        # 
        out = self.fc(out)  # 让输入和输出一致，方便残差连接
        out = self.dropout(out)  # 训练阶段随机丢弃一部分神经元，防止过拟合
        # 残差连接（out+Q out是残差）+layernorm
        return self.norm(out+Q), attn  # 返回输出和注意力权重

#### 前馈神经网络

In [9]:
class FeedForward(nn.Module):
    # d_ff: 线性层的扩展维度
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)  # 输入维度为d_model, 输出为d_ff, 512->2048, 为了让模型学到一个更丰富的特征
        self.fc2 = nn.Linear(d_ff, d_model)  # 保证第二个线性层输出维度等于第一个线性层的输入维度，为了后续做残差连接
        self.dropout = nn.Dropout(dropout)  # 随机丢弃神经元，防止过拟合
        self.norm = nn.LayerNorm(d_model)  # 对最后一维进行归一化
    
    def forward(self, x):
        # X: batch, seq_len, d_model
        # fc1->relu->dropout->fc2
        out = self.fc2(self.dropout(torch.relu(self.fc(x))))
        return self.norm(out+x)

#### 编码层

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        # 多头注意力机制
        # 输入为src, 实现序列内部的信息交互，每个token都可以看到序列中的其他token, 从而可以学到上下文依赖
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        # 前馈神经网络: 多每个位置向量独立进行非线性变换，提升模型表达能力
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, src, src_mask):
        '''
        input -> MultiHeadAttention -> FeedForward
        src 输入序列张量 形状：batch, seq_len, d_model
        因为是自注意力机制，所以 q, k, v都是同一个src
        src_mask 屏蔽padding的位置，避免模型关注无效token(encoder),在decoder中mask用来防止看到未来的词
        '''
        out, _ = self.self_attn(src, src, src, src_mask)
        # 经过前馈神经网络，每个位置的token都会单独通过两层线性层和激活函数，提升模型的表达能力
        out = self.ffn(out)
        # 返回编码后的结果
        return out

#### 解码层

In [11]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        # 带有掩码的多头注意力机制
        # 输入 tgt 目标序列，在翻译任务中，为已经生成的前几个单词
        # 计算目标序列内部的自注意力，通过mask遮挡住未来的token
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        # 交叉注意力，和encoder做交互
        # 输入 Q = 当前解码器的输出，K=V=来自编码器的memory（原序列上下文信息）
        # 为了让目标序列与原序列进行对齐
        self.cross_atten = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)  # 为了提升模型的表达能力

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        '''
        tgt: 目标序列
        memory: 编码器输出(原序列的表示)
        tgt_mask: 屏蔽未来的token
        memory_mask: 对padding进行掩码，防止模型学习无效token
        '''
        # 目标序列内部的自注意力，未来位置被mask
        out, _ = self.self_attn(tgt, tgt, tgt, tgt_mask)
        # 将目标序列 和 原序列 进行交互，Q为解码器当前的输出out, K=V=memory(编码器的输出)
        out, _ = self.cross_atten(out, memory, memory, memory_mask)
        out = self.ffn(out)
        return out

#### 位置编码

In [13]:
class PositionEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        '''
        d_model: 每个词向量的维度
        max_len: 句子的最大长度
        '''
        # 初始化位置编码矩阵，形状为max_len, d_model
        pe = torch.zeros(max_len, d_model)

        # 定义记录每个token位置的索引 0~max_len-1
        # (max_len, 1) 方便后续与缩放因子进行相乘
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # div_term 每个维度得到缩放因子
        # torch.arange(0, d_model, 2): 生成偶数维度索引 0, 2, 4.....
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        # 每个token的位置索引 * 每个维度的缩放因子（div_term）,在套上sin得到偶数维度的位置编码值
        pe[:, 0::2] = torch.sin(position * div_term)
        # 每个token的位置索引 * 每个维度的缩放因子（div_term）,在套上cos得到奇数维度的位置编码值
        pe[:, 1::2] = torch.cos(position*div_term)
        # 增加batch维度，1, max_len, d_model, 方便后续与输入embedding进行相加
        pe = pe.unsqueeze(0)
        # 注册为buffer，把位置编码pe存在模型里面，但不参与训练，随着模型保存/加载
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: 输入的embedding  形状：batch，seq_len，d_model
        seq_len = x.size(1)
        # 每个token的embedding加上对应的位置编码
        # self.pe[:, :seq_len, :] 取前seq_len个位置 形状：1, seq_len, d_model 可以与输入x对齐
        # embedding加上位置编码，transformer就能知道token的位置关系
        return x + self.pe[:, :seq_len, :]

#### 编码器

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_layers, d_ff, dropout=0.1, max_len=5000):
        super().__init__()
        # 词嵌入层，vocab_size: 词表大小，包含了不同token的总数
        # 将输入的token ID（对原始文本分词得到词表，不同词对应不同ID）转换成连续向量，维度为d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        # 位置编码 加入到序列中的token的位置信息
        self.pos_encoding = PositionEncoding(d_model, max_len)

        # 构建编码器的堆叠结构
        # 堆叠num_layers个encoder
        # nn.ModuleList 为网络层准备的列表 用来存放多个子模块
        # 列表推导式 用来生成num_layers个encoder
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, src, src_mask=None):
        # 将输入 token ID 转换成 embedding向量
        # 输出 形状 batch, seq_len, d_model
        # 乘上 sqrt(d_model),进行缩放，让后续注意力计算更加稳定
        out = self.embedding(src) * math.sqrt(self.embedding.embedding_dim)
        # 经过位置编码，添加位置信息
        out = self.pos_encoding(out)
        # 逐层经过encoderlayer
        for layer in self.layers:
            out = layer(out, src_mask)  # self_attn+ffn
        
        return out # 返回编码后的输出 batch, seq_len, d_model

#### 解码器

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_layers, d_ff, dropout=0.1, max_len=5000):
        super().__init__()
        # 将目标序列的token ID 转换为向量  维度为d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        # 进行位置编码
        self.pos_encoding = PositionEncoding(d_model, max_len)
        # 定义解码器列表
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        # 输出投影层 将decoder的输出映射回原词表的大小，从而得到每个 token 的预测分布
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        # tgt 目标序列 解码器的输入
        # memory 编码器的输出 也叫上下文信息
        # tgt_mask 目标序列的mask 用来屏蔽未来的位置
        # memory_mask 用来屏蔽pad
        out = self.embedding(tgt) * math.sqrt(self.embedding.embedding_dim)
        # 添加位置编码
        out = self.pos_encoding(out)
        # 逐层经过 decoder_layer
        for layer in self.layers:
            out = layer(out, memory, tgt_mask, memory_mask)
        # 将解码器最后一层输出的隐藏向量映射回原词汇表的维度，得到每个token的预测向量
        return self.fc_out(out)

#### Transformer整体架构

In [16]:
class Transformer(nn.Module):
    def __init__(self, 
                 src_vocab,  # 原语言词表大小  
                 tgt_vocab,  # 目标语言词表大小
                 d_model=512,  # embedding向量的维度
                 n_heads=8,  # 多头注意力的头数
                 num_encoder_layers=6,  # 编码层的层数
                 num_decoder_layers=6,  # 解码器的层数
                 d_ff=2048,  # ffn隐藏层维度
                 dropout=0.1,  # 丢弃比例
                 max_len=5000):  # 最大序列长度   
        super().__init__()
        # 编码器
        self.encoder = Encoder(src_vocab, d_model, n_heads, num_encoder_layers, d_ff, dropout, max_len)
        # 解码器
        self.decoder = Decoder(tgt_vocab, d_model, n_heads, num_decoder_layers, d_ff, dropout, max_len)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None):
        # 编码器前向传播 src_mask来屏蔽pad
        memory = self.encoder(src, src_mask)
        # 解码器前向传播 tgt_mask用来屏蔽未来token
        out = self.decoder(tgt, memory, tgt_mask, memory_mask)
        # 返回transformer输出 batch, seq_len_tgt, tgt_vocab
        return out

#### Mask-Attention机制

In [ ]:
def generate_mask(size):
    # size为序列长度
    # 生成一个上三角，不包含对角线，也就是需要屏蔽的位置
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    # 使用mask==0进行取反操作，得到需要被看见的位置
    return mask==0  # True可见，False屏蔽 